<a href="https://colab.research.google.com/github/eishaiqbal767-cmyk/AI-TMBD-MOVIES-PROJECT/blob/main/Notebooks/TMBD_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
BASE = '/content/drive/MyDrive/AI_Project'
print(os.path.exists(BASE))
print(os.listdir(BASE))

True
['notebooks', 'data', 'reports', 'src']


I mounted my Google Drive in Colab so the notebook can read and save files in my project folder.

In [ ]:
import requests
from google.colab import userdata

TOKEN = userdata.get('TMDB_TOKEN')
BASE_URL = 'https://api.themoviedb.org/3'
HEADERS = {
    'Authorization': f'Bearer {TOKEN}',
    'accept': 'application/json'
}

params = {
    'primary_release_date.gte': '2016-01-01',
    'primary_release_date.lte': '2016-12-31',
    'sort_by': 'vote_count.desc',
    'page': 1
}
r = requests.get(f'{BASE_URL}/discover/movie', headers=HEADERS, params=params)
print('Status code:', r.status_code)
data = r.json()
print('Total results:', data['total_results'])
print('Total pages:', data['total_pages'])
print('Movies on this page:', len(data['results']))
print('First movie:', data['results'][0]['title'], '| id:', data['results'][0]['id'])

Status code: 200
Total results: 20001
Total pages: 1001
Movies on this page: 20
First movie: Deadpool | id: 293660


I made a first test request to the TMDB Discover endpoint to check that my API token works.

# TMDB Movie Data Analysis and Machine Learning

Programming for AI, semester project. Team: [Eisha Iqbal,25L-3003], [Ifrah noor, 25L-0025]. Instructor: Mahhek Tahir.

In this notebook I collect movie data from the TMDB API for the years 2016 to 2026, clean and explore it, and build regression models to predict a numerical target.

In [ ]:
import os, re, json, time, hashlib, random
from datetime import datetime
import numpy as np
import pandas as pd

ROLL_NUMBERS = ['25L-3003', '25L-0025']

SEED = 25

random.seed(SEED)
np.random.seed(SEED)

BASE = '/content/drive/MyDrive/AI_Project'
RAW_DIR = f'{BASE}/data/raw'
PROC_DIR = f'{BASE}/data/processed'
JSON_DIR = f'{RAW_DIR}/json'
os.makedirs(JSON_DIR, exist_ok=True)

START_YEAR, END_YEAR = 2016, 2026
MOVIES_PER_YEAR = 100
SORT_BY = 'vote_count.desc'

print('Roll numbers:', ROLL_NUMBERS)
print('SEED:', SEED)
print('Raw data folder:', RAW_DIR)
print('Years:', START_YEAR, 'to', END_YEAR, '| target per year:', MOVIES_PER_YEAR)
print('Sorting:', SORT_BY)

Roll numbers: ['25L-3003', '25L-0025']
SEED: 25
Raw data folder: /content/drive/MyDrive/AI_Project/data/raw
Years: 2016 to 2026 | target per year: 100
Sorting: vote_count.desc


In [ ]:
COLLECTION_START = datetime.now()
request_count = 0
id_records = []
seen_ids = set()

def api_get(path, params=None):
    global request_count
    for attempt in range(5):
        r = requests.get(f'{BASE_URL}{path}', headers=HEADERS, params=params, timeout=30)
        request_count += 1
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            time.sleep(2 * (attempt + 1))
            continue
        r.raise_for_status()
    raise RuntimeError('Too many failed attempts')

year_summary = []
for year in range(START_YEAR, END_YEAR + 1):
    collected, page, total_pages = 0, 1, None
    while collected < MOVIES_PER_YEAR:
        params = {
            'primary_release_date.gte': f'{year}-01-01',
            'primary_release_date.lte': f'{year}-12-31',
            'sort_by': SORT_BY,
            'vote_count.gte': 10,
            'include_adult': 'false',
            'language': 'en-US',
            'page': page
        }
        data = api_get('/discover/movie', params)
        total_pages = data['total_pages']
        for m in data['results']:
            if m['id'] in seen_ids:
                continue
            seen_ids.add(m['id'])
            id_records.append({'id': m['id'], 'title': m['title'],
                               'discover_year': year, 'discover_page': page})
            collected += 1
            if collected >= MOVIES_PER_YEAR:
                break
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.05)
    year_summary.append({'year': year, 'collected': collected,'pages_used': page - 1
                         if collected >= MOVIES_PER_YEAR else page,
                         'api_total_pages': total_pages})

    print(year, '->', collected, 'movies')

ids_df = pd.DataFrame(id_records)
ids_df.to_csv(f'{RAW_DIR}/movie_ids.csv', index=False)
pd.DataFrame(year_summary).to_csv(f'{RAW_DIR}/discover_summary.csv', index=False)

print('\nUnique IDs:', ids_df['id'].nunique(), '| Requests so far:', request_count)
pd.DataFrame(year_summary)

2016 -> 100 movies
2017 -> 100 movies
2018 -> 100 movies
2019 -> 100 movies
2020 -> 100 movies
2021 -> 100 movies
2022 -> 100 movies
2023 -> 100 movies
2024 -> 100 movies
2025 -> 100 movies
2026 -> 100 movies

Unique IDs: 1100 | Requests so far: 55


,year,collected,pages_used,api_total_pages
0,2016,100,5,173
1,2017,100,5,185
2,2018,100,5,186
3,2019,100,5,184
4,2020,100,5,142
5,2021,100,5,148
6,2022,100,5,160
7,2023,100,5,153
8,2024,100,5,129
9,2025,100,5,102


The Discover step returned 1100 unique movie IDs, exactly 100 for each year from 2016 to 2026, with no duplicates. Each year needed 5 pages of 20 results, and the 11 years together used 55 requests.


In [ ]:
total = len(ids_df)
done_now, skipped = 0, 0

for i, movie_id in enumerate(ids_df['id'], start=1):
    path = f'{JSON_DIR}/{movie_id}.json'
    if os.path.exists(path):
        skipped += 1
        continue
    data = api_get(f'/movie/{movie_id}',
                   {'append_to_response': 'credits', 'language': 'en-US'})
    wrapper = {'collected_at': datetime.now().isoformat(timespec='seconds'),
               'data': data}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(wrapper, f)
    done_now += 1
    if i % 100 == 0:
        print(f'{i}/{total} processed')
    time.sleep(0.05)

COLLECTION_END = datetime.now()
files = [f for f in os.listdir(JSON_DIR) if f.endswith('.json')]
print('\nDownloaded now:', done_now, '| skipped (already saved):', skipped)
print('JSON files on Drive:', len(files))
print('Total requests (discover + details):', request_count)
print('Start:', COLLECTION_START.isoformat(timespec='seconds'))
print('End:  ', COLLECTION_END.isoformat(timespec='seconds'))

with open(f'{RAW_DIR}/collection_log.json', 'w') as f:
    json.dump({'roll_numbers': ROLL_NUMBERS,
               'start': COLLECTION_START.isoformat(timespec='seconds'),
               'end': COLLECTION_END.isoformat(timespec='seconds'),
               'requests': request_count}, f)

100/1100 processed
200/1100 processed
300/1100 processed
400/1100 processed
500/1100 processed
600/1100 processed
700/1100 processed
800/1100 processed
900/1100 processed
1000/1100 processed
1100/1100 processed

Downloaded now: 1100 | skipped (already saved): 0
JSON files on Drive: 1100
Total requests (discover + details): 1155
Start: 2026-11-21T19:51:32
End:   2026-11-21T20:07:55


For each of the 1100 movie IDs I called the Movie Details endpoint and asked for the credits in the same request, so I got budget, revenue, runtime, genres, languages, production companies, cast and crew without extra calls.